GDF-Net Scale-Anchored Bi-ALT, Weight-Space Model Soup, Real-Time Distillation, and 8-View GeM Ensemble on ISIC-2018
Dataset: ISIC-2018 Task 1 (2,594 train / 100 validation / 1,000 test images at 384x384).
Test results (Table 1 Panel B and Table 3 Panel B): GDF-Net (+Bi-ALT, T_base=0.74) achieves 0.9105 Dice; GDF-Net (Soup, SV No KD) achieves 0.9126 Dice; GDF-Net (Real-Time, SV KD) achieves 0.9127 Dice with 74 low-IoU cases at 31.3 FPS; GDF-Net (Ensemble, 8V MS-TTA) achieves 0.9146 Dice and 0.8554 mIoU.


Scale-Anchored Bi-ALT calibration, weight-space model soup, 8-view GeM MS-TTA, and 15-epoch online 4-teacher knowledge distillation.


In [1]:
"""
========================================================================================
Phase 28 (Proposal GDF-Net Weight-Soup + GDF-Net Real-Time KD Combined):
Ultimate Single-Model 100% Pure-CNN Optimization (Strictly 1 Model @ Test Time, 29.50M Params)
  1. GDF-Net Weight-Soup (Zero-Retrain Single-Model Enhancements):
     - 3-Way Stage-Decoupled Weight Soup (combining M1 300ep Canonical, M2 60ep LLRD-MS,
       and M4 20ep Asym-Tversky in parameter space with separate Encoder/Decoder weights)
     - Adaptive BatchNorm Running-Stat Recalibration (AdaBN)
     - Single-Model 8-View Multi-Scale GeM TTA ({352, 384, 448} with Power-Mean p)
     - Scale-Anchored Bidirectional Adaptive Lesion Thresholding (Bi-ALT, T_base in [0.35, 0.93])
       + Morphological LCC Post-Processing
  2. GDF-Net Real-Time KD (4-Teacher (M1-M4) Ensemble-to-Single Self-Distillation):
     - Uses the 0.9123-Dice PA1 Pure-CNN Ensemble (M1 + M2 + M3 FeedbackModulator + M4)
       as a Frozen Online Soft-Probability & Logit Teacher
     - Distills knowledge into a SINGLE CanonicalGDFNet (ConvFormer-S18, 29.50M) student
     - Applies Distilled Weight-Space Ensembling (Distilled-WiSE-FT) on Official Validation
========================================================================================
"""

import os
import sys
import glob
import time
import copy
import json
import random
import zipfile
import shutil
import subprocess
import numpy as np
import pandas as pd
from PIL import Image
import cv2

from scipy.ndimage import distance_transform_edt, binary_erosion, binary_fill_holes

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import timm
import albumentations as A
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

BACKBONE_NAME = "convformer_s18.sail_in22k_ft_in1k_384"
MODEL_TAG = "ConvFormerS18_PA2Plus_Distill"
DISTILL_EPOCHS = 15
IMG_SIZE = 384
BATCH_SIZE = 16


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)


# ====================================================================
# 1A. Canonical GDF-Net Pure-CNN Architecture (Student & M1/M2/M4, 29.50M)
# ====================================================================
class HierarchicalBackboneV4(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, pretrained=False, in_chans=3, drop_path_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            in_chans=in_chans,
            features_only=True,
            out_indices=(0, 1, 2, 3),
            drop_path_rate=drop_path_rate
        )
        self.channels = self.backbone.feature_info.channels()

    def get_channels(self):
        return tuple(self.channels)

    def forward(self, x):
        feats = self.backbone(x)
        return feats[0], feats[1], feats[2], feats[3]


class MultiScaleFrequencyFilterCanonical(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.in_channels = in_channels
        mid_channels = max(in_channels // reduction, 16)

        self.dw_conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=in_channels, bias=False)
        self.dw_conv3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False)
        self.dw_conv5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2, groups=in_channels, bias=False)

        self.pw_low = nn.Sequential(
            nn.Conv2d(in_channels * 3, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.hf_gate = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, groups=mid_channels, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        low1 = self.dw_conv1(x)
        low3 = self.dw_conv3(x)
        low5 = self.dw_conv5(x)
        x_low = self.pw_low(torch.cat([low1, low3, low5], dim=1))
        x_high = x - x_low
        gate = self.hf_gate(x_low)
        x_high_filtered = x_high * gate
        out = self.fusion(torch.cat([x_low, x_high_filtered], dim=1))
        return out + x


class MaskGuidedLGAG(nn.Module):
    def __init__(self, skip_channels, gating_channels, out_channels=None):
        super().__init__()
        out_channels = out_channels or skip_channels
        self.skip_channels = skip_channels
        self.gating_channels = gating_channels

        self.g_proj = nn.Sequential(
            nn.Conv2d(gating_channels, skip_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(skip_channels)
        )

        mid_ch = max(skip_channels // 2, 16)
        self.conv1x1 = nn.Conv2d(skip_channels + 1, mid_ch, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu = nn.ReLU(inplace=True)

        self.dw_dil1 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=1, dilation=1, groups=mid_ch, bias=False)
        self.dw_dil2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=2, dilation=2, groups=mid_ch, bias=False)
        self.bn_dw = nn.BatchNorm2d(mid_ch)

        self.attn_proj = nn.Sequential(
            nn.Conv2d(mid_ch, skip_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(skip_channels),
            nn.Sigmoid()
        )

        self.out_conv = nn.Sequential(
            nn.Conv2d(skip_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, skip, gating, mask_prob=None):
        B, C, H, W = skip.shape
        g_up = F.interpolate(gating, size=(H, W), mode='bilinear', align_corners=False)
        g_proj = self.g_proj(g_up)

        if mask_prob is not None:
            m_up = F.interpolate(mask_prob, size=(H, W), mode='bilinear', align_corners=False)
        else:
            m_up = torch.zeros((B, 1, H, W), device=skip.device, dtype=skip.dtype)

        feat = skip + g_proj
        feat_with_mask = torch.cat([feat, m_up], dim=1)

        x = self.relu(self.bn1(self.conv1x1(feat_with_mask)))
        x = self.relu(self.bn_dw(self.dw_dil1(x) + self.dw_dil2(x)))
        attn = self.attn_proj(x)

        gated_skip = skip * attn
        return self.out_conv(gated_skip)


class MultiScaleConvolutionalAttentionModule(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.dw1 = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=in_channels, bias=False)
        self.dw3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False)
        self.dw5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2, groups=in_channels, bias=False)
        self.bn_dw = nn.BatchNorm2d(in_channels * 3)

        self.pw = nn.Sequential(
            nn.Conv2d(in_channels * 3, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        mid_ch = max(out_channels // 4, 16)
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels, mid_ch, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, out_channels, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

        self.sa = nn.Sequential(
            nn.Conv2d(out_channels, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        d1 = self.dw1(x)
        d3 = self.dw3(x)
        d5 = self.dw5(x)
        dw_cat = F.relu(self.bn_dw(torch.cat([d1, d3, d5], dim=1)), inplace=True)
        feat = self.pw(dw_cat)
        feat = feat * self.ca(feat)
        feat = feat * self.sa(feat)
        return self.refine(feat)


class CascadedMSCAMDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.mscam = MultiScaleConvolutionalAttentionModule(in_channels + skip_channels, out_channels)
        self.head = nn.Conv2d(out_channels, 1, kernel_size=1)

    def forward(self, x, gated_skip):
        x_up = self.upsample(x)
        if x_up.shape[2:] != gated_skip.shape[2:]:
            x_up = F.interpolate(x_up, size=gated_skip.shape[2:], mode='bilinear', align_corners=False)
        fused = torch.cat([x_up, gated_skip], dim=1)
        out_feat = self.mscam(fused)
        pred_logits = self.head(out_feat)
        return out_feat, pred_logits


class CanonicalGDFNet(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, pretrained=False, in_chans=3, decoder_channels=64, drop_path_rate=0.2):
        super().__init__()
        self.backbone_name = backbone_name
        self.decoder_channels = decoder_channels

        self.backbone = HierarchicalBackboneV4(
            backbone_name=backbone_name,
            pretrained=pretrained,
            in_chans=in_chans,
            drop_path_rate=drop_path_rate
        )
        c1, c2, c3, c4 = self.backbone.get_channels()

        self.fdf1 = MultiScaleFrequencyFilterCanonical(c1)
        self.fdf2 = MultiScaleFrequencyFilterCanonical(c2)
        self.fdf3 = MultiScaleFrequencyFilterCanonical(c3)
        self.fdf4 = MultiScaleFrequencyFilterCanonical(c4)

        dec4_ch = decoder_channels * 4
        self.dec4_proj = nn.Sequential(
            nn.Conv2d(c4, dec4_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec4_ch),
            nn.ReLU(inplace=True)
        )
        self.head4 = nn.Conv2d(dec4_ch, 1, kernel_size=1)

        dec3_ch = decoder_channels * 3
        dec2_ch = decoder_channels * 2
        dec1_ch = decoder_channels

        self.gate3 = MaskGuidedLGAG(skip_channels=c3, gating_channels=dec4_ch, out_channels=c3)
        self.gate2 = MaskGuidedLGAG(skip_channels=c2, gating_channels=dec3_ch, out_channels=c2)
        self.gate1 = MaskGuidedLGAG(skip_channels=c1, gating_channels=dec2_ch, out_channels=c1)

        self.dec3 = CascadedMSCAMDecoderBlock(in_channels=dec4_ch, skip_channels=c3, out_channels=dec3_ch)
        self.dec2 = CascadedMSCAMDecoderBlock(in_channels=dec3_ch, skip_channels=c2, out_channels=dec2_ch)
        self.dec1 = CascadedMSCAMDecoderBlock(in_channels=dec2_ch, skip_channels=c1, out_channels=dec1_ch)

        self.final_refine = nn.Sequential(
            nn.Conv2d(dec1_ch, dec1_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec1_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(dec1_ch, 1, kernel_size=1)
        )

    def forward(self, x, return_deep_supervision=False):
        B, C, H, W = x.shape
        f1, f2, f3, f4 = self.backbone(x)

        s1 = self.fdf1(f1)
        s2 = self.fdf2(f2)
        s3 = self.fdf3(f3)
        s4 = self.fdf4(f4)

        d4 = self.dec4_proj(s4)
        p4 = self.head4(d4)
        m4_prob = torch.sigmoid(p4)

        gated_s3 = self.gate3(s3, d4, m4_prob)
        d3, p3 = self.dec3(d4, gated_s3)
        m3_prob = torch.sigmoid(p3)

        gated_s2 = self.gate2(s2, d3, m3_prob)
        d2, p2 = self.dec2(d3, gated_s2)
        m2_prob = torch.sigmoid(p2)

        gated_s1 = self.gate1(s1, d2, m2_prob)
        d1, p1 = self.dec1(d2, gated_s1)

        p1_up = F.interpolate(p1, size=(H, W), mode='bilinear', align_corners=False)
        p2_up = F.interpolate(p2, size=(H, W), mode='bilinear', align_corners=False)
        p3_up = F.interpolate(p3, size=(H, W), mode='bilinear', align_corners=False)
        p4_up = F.interpolate(p4, size=(H, W), mode='bilinear', align_corners=False)

        d1_up = F.interpolate(d1, size=(H, W), mode='bilinear', align_corners=False)
        refined_logits = self.final_refine(d1_up)

        out_final = refined_logits + p1_up + p2_up + p3_up + p4_up
        if return_deep_supervision:
            return out_final, p1_up, p2_up, p3_up, p4_up
        return out_final


# ====================================================================
# 1B. FeedbackGDFNet (Used ONLY inside Frozen 4-Teacher (M1-M4) during Training)
# ====================================================================
class MultiScaleFrequencyFilterFeedback(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.in_channels = in_channels
        mid_channels = max(in_channels // reduction, 16)

        self.dw_conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=in_channels, bias=False)
        self.dw_conv3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False)
        self.dw_conv5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2, groups=in_channels, bias=False)

        self.pw_low = nn.Sequential(
            nn.Conv2d(in_channels * 3, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.hf_gate = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, groups=mid_channels, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        self.gamma = nn.Parameter(torch.ones(1, in_channels, 1, 1) * 0.5)

    def forward(self, x):
        low_1 = self.dw_conv1(x)
        low_3 = self.dw_conv3(x)
        low_5 = self.dw_conv5(x)
        f_low = self.pw_low(torch.cat([low_1, low_3, low_5], dim=1))
        f_high_raw = x - f_low
        gate = self.hf_gate(f_high_raw)
        f_high = f_high_raw * gate
        out = self.fusion(torch.cat([f_low, f_high], dim=1))
        return x + self.gamma * out


class AttentionGate(nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(f_g, f_int, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(f_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(f_l, f_int, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(f_int)
        )
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(f_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.Sigmoid()
        )

    def forward(self, g, x):
        if g.shape[2:] != x.shape[2:]:
            g = F.interpolate(g, size=x.shape[2:], mode='bilinear', align_corners=False)
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        alpha = self.psi(g1 + x1)
        return x * alpha


class FeedbackModulator(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv_gate = nn.Sequential(
            nn.Conv2d(channels + 1, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=1, bias=True),
            nn.Sigmoid()
        )
        self.alpha = nn.Parameter(torch.ones(1, channels, 1, 1) * 0.3)

    def forward(self, feat, coarse_prob):
        if coarse_prob.shape[2:] != feat.shape[2:]:
            coarse_prob = F.interpolate(coarse_prob, size=feat.shape[2:], mode='bilinear', align_corners=False)
        gate = self.conv_gate(torch.cat([feat, coarse_prob], dim=1))
        return feat * (1.0 + self.alpha * gate)


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class DeepSupervisionDecoder(nn.Module):
    def __init__(self, c1, c2, c3, c4, num_classes=1):
        super().__init__()
        self.ag3 = AttentionGate(f_g=c4, f_l=c3, f_int=c3 // 2)
        self.ag2 = AttentionGate(f_g=c3, f_l=c2, f_int=c2 // 2)
        self.ag1 = AttentionGate(f_g=c2, f_l=c1, f_int=c1 // 2)

        self.dec4 = ConvBlock(c4, c4)
        self.dec3 = ConvBlock(c4 + c3, c3)
        self.dec2 = ConvBlock(c3 + c2, c2)
        self.dec1 = ConvBlock(c2 + c1, c1)

        self.head4 = nn.Conv2d(c4, num_classes, kernel_size=1)
        self.head3 = nn.Conv2d(c3, num_classes, kernel_size=1)
        self.head2 = nn.Conv2d(c2, num_classes, kernel_size=1)
        self.head1 = nn.Conv2d(c1, num_classes, kernel_size=1)

        self.final_refine = nn.Sequential(
            nn.Conv2d(c1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, kernel_size=1)
        )

    def forward(self, f1, f2, f3, f4, target_size):
        d4 = self.dec4(f4)
        d4_up = F.interpolate(d4, size=f3.shape[2:], mode='bilinear', align_corners=False)
        f3_att = self.ag3(g=d4_up, x=f3)
        d3 = self.dec3(torch.cat([d4_up, f3_att], dim=1))

        d3_up = F.interpolate(d3, size=f2.shape[2:], mode='bilinear', align_corners=False)
        f2_att = self.ag2(g=d3_up, x=f2)
        d2 = self.dec2(torch.cat([d3_up, f2_att], dim=1))

        d2_up = F.interpolate(d2, size=f1.shape[2:], mode='bilinear', align_corners=False)
        f1_att = self.ag1(g=d2_up, x=f1)
        d1 = self.dec1(torch.cat([d2_up, f1_att], dim=1))

        d1_full = F.interpolate(d1, size=target_size, mode='bilinear', align_corners=False)
        logits = self.final_refine(d1_full)
        return logits


class FeedbackGDFNet(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, pretrained=False, num_classes=1, drop_path_rate=0.0):
        super().__init__()
        self.backbone = HierarchicalBackboneV4(
            backbone_name=backbone_name, pretrained=pretrained, in_chans=3, drop_path_rate=drop_path_rate
        )
        c1, c2, c3, c4 = self.backbone.get_channels()

        self.freq1 = MultiScaleFrequencyFilterFeedback(c1)
        self.freq2 = MultiScaleFrequencyFilterFeedback(c2)
        self.freq3 = MultiScaleFrequencyFilterFeedback(c3)
        self.freq4 = MultiScaleFrequencyFilterFeedback(c4)

        self.fb1 = FeedbackModulator(c1)
        self.fb2 = FeedbackModulator(c2)
        self.fb3 = FeedbackModulator(c3)

        self.decoder = DeepSupervisionDecoder(c1, c2, c3, c4, num_classes=num_classes)

    def forward(self, x):
        target_size = x.shape[2:]
        s1, s2, s3, s4 = self.backbone(x)

        f1 = self.freq1(s1)
        f2 = self.freq2(s2)
        f3 = self.freq3(s3)
        f4 = self.freq4(s4)

        coarse_logits = self.decoder.head4(self.decoder.dec4(f4))
        coarse_prob = torch.sigmoid(coarse_logits)

        f1_mod = self.fb1(f1, coarse_prob)
        f2_mod = self.fb2(f2, coarse_prob)
        f3_mod = self.fb3(f3, coarse_prob)

        return self.decoder(f1_mod, f2_mod, f3_mod, f4, target_size=target_size)


# ====================================================================
# 2. GDF-Net Real-Time KD Knowledge Distillation + Mild Asymmetric Tversky Loss
# ====================================================================
class EnsembleDistillationLoss(nn.Module):
    def __init__(self, alpha_fp=0.42, beta_fn=0.58, gamma=1.25, pos_weight=1.18, kd_weight=0.55, temp=2.0):
        super().__init__()
        self.alpha_fp = alpha_fp
        self.beta_fn = beta_fn
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.kd_weight = kd_weight
        self.temp = temp

    def _single_scale_gt_loss(self, pred_logits, target_mask):
        edge = torch.abs(target_mask - F.avg_pool2d(target_mask, kernel_size=31, stride=1, padding=15))
        weight = 1.0 + 4.0 * edge

        pw = torch.tensor([self.pos_weight], device=pred_logits.device, dtype=pred_logits.dtype)
        bce = F.binary_cross_entropy_with_logits(pred_logits, target_mask, pos_weight=pw, reduction='none')
        wbce = (weight * bce).sum(dim=(2, 3)) / (weight.sum(dim=(2, 3)) + 1e-8)

        pred = torch.sigmoid(pred_logits)
        smooth = 1e-5

        tp = (pred * target_mask).sum(dim=(1, 2, 3))
        fp = (pred * (1.0 - target_mask)).sum(dim=(1, 2, 3))
        fn = ((1.0 - pred) * target_mask).sum(dim=(1, 2, 3))

        tversky = (tp + smooth) / (tp + self.alpha_fp * fp + self.beta_fn * fn + smooth)
        asym_focal_tversky = torch.pow(1.0 - tversky, self.gamma)

        macro_dice_loss = 1.0 - (2.0 * tp + smooth) / (2.0 * tp + fp + fn + smooth)
        return (0.30 * wbce.squeeze(1) + 0.35 * asym_focal_tversky + 0.35 * macro_dice_loss).mean()

    def _kd_loss(self, student_logits, teacher_prob, teacher_logits, target_mask):
        pred_s = torch.sigmoid(student_logits)
        smooth = 1e-5

        inter = (pred_s * teacher_prob).sum(dim=(1, 2, 3))
        denom = (pred_s + teacher_prob).sum(dim=(1, 2, 3))
        soft_dice_kd = (1.0 - (2.0 * inter + smooth) / (denom + smooth)).mean()

        edge = torch.abs(teacher_prob - F.avg_pool2d(teacher_prob, kernel_size=15, stride=1, padding=7))
        w = 1.0 + 3.0 * edge
        soft_bce = F.binary_cross_entropy_with_logits(student_logits, teacher_prob, reduction='none')
        wbce_kd = ((w * soft_bce).sum(dim=(2, 3)) / (w.sum(dim=(2, 3)) + 1e-8)).mean()

        p_s_temp = torch.sigmoid(student_logits / self.temp)
        p_t_temp = torch.sigmoid(teacher_logits / self.temp)
        temp_mse = F.mse_loss(p_s_temp, p_t_temp) * (self.temp ** 2)

        return 0.45 * soft_dice_kd + 0.35 * wbce_kd + 0.20 * temp_mse

    def forward(self, outputs, target, teacher_prob, teacher_logits):
        out_final, p1, p2, p3, p4 = outputs
        loss_gt = (
            1.0 * self._single_scale_gt_loss(out_final, target) +
            0.4 * self._single_scale_gt_loss(p1, target) +
            0.3 * self._single_scale_gt_loss(p2, target) +
            0.2 * self._single_scale_gt_loss(p3, target) +
            0.1 * self._single_scale_gt_loss(p4, target)
        ) / 2.0

        loss_kd = (
            0.75 * self._kd_loss(out_final, teacher_prob, teacher_logits, target) +
            0.25 * self._kd_loss(p1, teacher_prob, teacher_logits, target)
        )
        return (1.0 - self.kd_weight) * loss_gt + self.kd_weight * loss_kd


class ModelEMA:
    def __init__(self, model, decay=0.996):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1.0 - self.decay)
            else:
                v.copy_(msd[k])


# ====================================================================
# 3. Dataset, SSD Caching & Metrics
# ====================================================================
class FastISICDataset(Dataset):
    def __init__(self, imgs, msks, is_train=False):
        self.imgs = imgs
        self.msks = msks
        if is_train:
            self.transform = A.Compose([
                A.Resize(height=IMG_SIZE, width=IMG_SIZE),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.ShiftScaleRotate(
                    shift_limit=0.06, scale_limit=0.12, rotate_limit=30,
                    border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.6
                ),
                A.OneOf([
                    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.06, p=1.0),
                    A.RandomBrightnessContrast(brightness_limit=0.18, contrast_limit=0.18, p=1.0),
                    A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=1.0),
                ], p=0.45),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])
        else:
            self.transform = A.Compose([
                A.Resize(height=IMG_SIZE, width=IMG_SIZE),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs[idx], cv2.IMREAD_COLOR)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img = np.array(Image.open(self.imgs[idx]).convert("RGB"))
        msk = cv2.imread(self.msks[idx], cv2.IMREAD_GRAYSCALE)
        if msk is None:
            msk = np.array(Image.open(self.msks[idx]).convert("L"))

        augmented = self.transform(image=img, mask=msk)
        img_t = torch.from_numpy(augmented["image"].transpose(2, 0, 1)).float()
        msk_t = torch.from_numpy((augmented["mask"] > 127).astype(np.float32)).unsqueeze(0)
        return img_t, msk_t, os.path.basename(self.imgs[idx])


def prepare_fast_ssd_cache(imgs, msks, cache_dir, target_size=IMG_SIZE, desc="Dataset"):
    os.makedirs(cache_dir, exist_ok=True)
    cached_imgs, cached_msks = [], []
    need_build = False
    for im_p, mk_p in zip(imgs, msks):
        base_id = os.path.splitext(os.path.basename(im_p))[0]
        c_im = os.path.join(cache_dir, f"{base_id}.jpg")
        c_mk = os.path.join(cache_dir, f"{base_id}_mask.png")
        cached_imgs.append(c_im)
        cached_msks.append(c_mk)
        if not (os.path.exists(c_im) and os.path.exists(c_mk)):
            need_build = True

    if need_build:
        print(f"[*] Caching {desc} ({len(imgs)} pairs -> {target_size}x{target_size})...", flush=True)
        t_cache = time.time()
        for im_p, mk_p, c_im, c_mk in zip(imgs, msks, cached_imgs, cached_msks):
            if not (os.path.exists(c_im) and os.path.exists(c_mk)):
                img = cv2.imread(im_p, cv2.IMREAD_COLOR)
                if img is None:
                    img = cv2.cvtColor(np.array(Image.open(im_p).convert('RGB')), cv2.COLOR_RGB2BGR)
                msk = cv2.imread(mk_p, cv2.IMREAD_GRAYSCALE)
                if msk is None:
                    msk = np.array(Image.open(mk_p).convert('L'))
                img_r = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)
                msk_r = cv2.resize(msk, (target_size, target_size), interpolation=cv2.INTER_NEAREST)
                cv2.imwrite(c_im, img_r, [int(cv2.IMWRITE_JPEG_QUALITY), 96])
                cv2.imwrite(c_mk, msk_r)
        print(f"[+] Cache ready in {time.time() - t_cache:.1f}s!", flush=True)
    return cached_imgs, cached_msks


def post_process_prediction(pred_bin):
    pred_u8 = (pred_bin > 0).astype(np.uint8)
    if pred_u8.sum() == 0:
        return pred_u8
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(pred_u8, connectivity=8)
    if num_labels <= 2:
        cleaned = pred_u8
    else:
        areas = stats[1:, cv2.CC_STAT_AREA]
        max_area = np.max(areas)
        cleaned = np.zeros_like(pred_u8)
        for label_idx, area in enumerate(areas, start=1):
            if area >= max_area * 0.08:
                cleaned[labels == label_idx] = 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    return cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)


def compute_surface_distances(pred_mask, gt_mask):
    pred_b = pred_mask.astype(bool)
    gt_b = gt_mask.astype(bool)
    if not np.any(pred_b) and not np.any(gt_b):
        return 0.0, 0.0, 1.0
    if not np.any(pred_b) or not np.any(gt_b):
        diag = float(np.sqrt(pred_b.shape[0] ** 2 + pred_b.shape[1] ** 2))
        return diag, diag / 2.0, 0.0

    pred_border = np.logical_xor(pred_b, binary_erosion(pred_b))
    gt_border = np.logical_xor(gt_b, binary_erosion(gt_b))
    if not np.any(pred_border):
        pred_border = pred_b
    if not np.any(gt_border):
        gt_border = gt_b

    dt_gt = distance_transform_edt(~gt_border)
    dt_pred = distance_transform_edt(~pred_border)
    d_pred_to_gt = dt_gt[pred_border]
    d_gt_to_pred = dt_pred[gt_border]

    all_d = np.concatenate([d_pred_to_gt, d_gt_to_pred])
    hd95 = float(np.percentile(all_d, 95))
    asd = float((d_pred_to_gt.mean() + d_gt_to_pred.mean()) / 2.0)

    tau = 2.0
    prec_b = float(np.mean(d_pred_to_gt <= tau))
    rec_b = float(np.mean(d_gt_to_pred <= tau))
    if prec_b + rec_b == 0:
        bf2 = 0.0
    else:
        bf2 = float(2.0 * prec_b * rec_b / (prec_b + rec_b + 1e-8))
    return hd95, asd, bf2


def find_checkpoints():
    ckpt_map = {
        "M1_Canonical_300ep": None,
        "M2_Canonical_60ep_MS": None,
        "M3_Feedback_25ep": None,
        "M4_Canonical_AsymTversky": None,
    }
    search_roots = ["/kaggle/input", "/kaggle/working", "./", "../"]
    all_pths = []
    for r in search_roots:
        if os.path.exists(r):
            for root, _, files in os.walk(r):
                for f in files:
                    if f.endswith(".pth"):
                        all_pths.append(os.path.join(root, f))

    print(f"[*] Found {len(all_pths)} .pth files across workspace:", flush=True)
    for p in all_pths:
        print(f"    -> {p}", flush=True)
        bn = os.path.basename(p)
        if bn in ("GDFNet_ConvFormerS18_ISIC2018_best.pth", "GDFNet_ConvFormerS18_300ep_best.pth"):
            ckpt_map["M1_Canonical_300ep"] = p
        elif bn == "GDFNet_ConvFormerS18_LLRD_MS_best.pth":
            ckpt_map["M2_Canonical_60ep_MS"] = p
        elif bn == "GDFNet_convformer_s18_best.pth":
            ckpt_map["M3_Feedback_25ep"] = p
        elif bn == "GDFNet_ConvFormerS18_AsymTversky_best.pth":
            ckpt_map["M4_Canonical_AsymTversky"] = p

    return ckpt_map


# ====================================================================
# 4. Single-Model Multi-Scale GeM TTA & Scale-Anchored Bi-ALT
# ====================================================================
@torch.no_grad()
def extract_single_model_views(model, loader, device):
    """
    Extracts 8 multi-scale/flip views for a SINGLE 29.50M Pure-CNN model:
      - View 0..3: 384x384 (Orig, HFlip, VFlip, HVFlip)
      - View 4..5: 448x448 (Orig, HFlip)
      - View 6..7: 352x352 (Orig, HFlip)
    """
    model.eval()
    v384_list, v448_list, v352_list = [], [], []
    gt_list, names_list = [], []

    for imgs, msks, names in loader:
        imgs = imgs.to(device, non_blocking=True)
        B, C, H, W = imgs.shape

        with autocast("cuda"):
            p0 = torch.sigmoid(model(imgs))
            p1 = torch.flip(torch.sigmoid(model(torch.flip(imgs, [3]))), [3])
            p2 = torch.flip(torch.sigmoid(model(torch.flip(imgs, [2]))), [2])
            p3 = torch.flip(torch.sigmoid(model(torch.flip(imgs, [2, 3]))), [2, 3])

            imgs_448 = F.interpolate(imgs, size=(448, 448), mode='bilinear', align_corners=False)
            p4 = F.interpolate(torch.sigmoid(model(imgs_448)), size=(H, W), mode='bilinear', align_corners=False)
            p5_flip = torch.sigmoid(model(torch.flip(imgs_448, [3])))
            p5 = F.interpolate(torch.flip(p5_flip, [3]), size=(H, W), mode='bilinear', align_corners=False)

            imgs_352 = F.interpolate(imgs, size=(352, 352), mode='bilinear', align_corners=False)
            p6 = F.interpolate(torch.sigmoid(model(imgs_352)), size=(H, W), mode='bilinear', align_corners=False)
            p7_flip = torch.sigmoid(model(torch.flip(imgs_352, [3])))
            p7 = F.interpolate(torch.flip(p7_flip, [3]), size=(H, W), mode='bilinear', align_corners=False)

        v384 = torch.stack([p0, p1, p2, p3], dim=1).squeeze(2).cpu().numpy().astype(np.float16)
        v448 = torch.stack([p4, p5], dim=1).squeeze(2).cpu().numpy().astype(np.float16)
        v352 = torch.stack([p6, p7], dim=1).squeeze(2).cpu().numpy().astype(np.float16)

        v384_list.append(v384)
        v448_list.append(v448)
        v352_list.append(v352)
        gt_list.append(msks.squeeze(1).numpy().astype(np.uint8))
        names_list.extend(names)

    return (
        np.concatenate(v384_list, axis=0),
        np.concatenate(v448_list, axis=0),
        np.concatenate(v352_list, axis=0),
        np.concatenate(gt_list, axis=0),
        names_list,
    )


def fuse_single_model_views(v384, v448, v352, mode="ms8_gem", gem_p=1.25, scale_weights=(4.0/6.0, 2.0/6.0, 0.0)):
    """
    Fuses views of a SINGLE 29.50M model using GeM power-mean.
    """
    if mode == "sv":
        return v384[:, 0].astype(np.float32)

    eps = 1e-6
    if mode == "tta4":
        v = np.clip(v384.astype(np.float32), eps, 1.0)
        if abs(gem_p - 1.0) < 1e-4:
            return np.mean(v, axis=1)
        return np.power(np.mean(np.power(v, gem_p), axis=1), 1.0 / gem_p)

    w384, w448, w352 = scale_weights
    v1 = np.clip(v384.astype(np.float32), eps, 1.0)
    v2 = np.clip(v448.astype(np.float32), eps, 1.0)
    v3 = np.clip(v352.astype(np.float32), eps, 1.0)

    if abs(gem_p - 1.0) < 1e-4:
        p384 = np.mean(v1, axis=1)
        p448 = np.mean(v2, axis=1)
        p352 = np.mean(v3, axis=1)
        return w384 * p384 + w448 * p448 + w352 * p352
    else:
        p384 = np.mean(np.power(v1, gem_p), axis=1)
        p448 = np.mean(np.power(v2, gem_p), axis=1)
        p352 = np.mean(np.power(v3, gem_p), axis=1)
        return np.power(w384 * p384 + w448 * p448 + w352 * p352, 1.0 / gem_p)


def compute_fast_val_dice(probs, gts, thresh_vec, use_post=False):
    if np.ndim(thresh_vec) == 0:
        preds = (probs > thresh_vec).astype(np.uint8)
    else:
        preds = (probs > thresh_vec[:, None, None]).astype(np.uint8)

    if use_post:
        for i in range(len(preds)):
            preds[i] = post_process_prediction(preds[i])

    p_f = preds.reshape(len(preds), -1)
    g_f = gts.reshape(len(gts), -1)
    inter = (p_f * g_f).sum(axis=1)
    union = p_f.sum(axis=1) + g_f.sum(axis=1)
    dices = np.where(union == 0, 1.0, (2.0 * inter + 1e-7) / (union + 1e-7))
    return float(np.mean(dices))


def get_bi_alt_thresholds(probs, alt_params):
    """
    Scale-Anchored Bidirectional ALT:
      - Anchors probe cutoffs (t_low, t_mid, t_high) to t_base so it generalizes
        across both standard models (t_base~0.74) and high-recall models (t_base~0.85-0.90).
      - Applies -dt_high to diffuse/giant melanomas (rho > 1.65 or area_mid > 0.45)
      - Applies -dt_mid to medium-uncertainty lesions (rho > 1.32 or area_mid > 0.28)
      - Applies +dt_crisp to crisp, well-bounded lesions
    """
    t_base = alt_params["t_base"]
    dt_high = alt_params["dt_high"]
    dt_mid = alt_params["dt_mid"]
    dt_crisp = alt_params.get("dt_crisp", 0.0)

    N, H, W = probs.shape
    total_px = float(H * W)
    t_vec = np.full(N, t_base, dtype=np.float32)

    t_low = float(np.clip(t_base - 0.39, 0.25, 0.65))
    t_mid = float(np.clip(t_base - 0.24, 0.35, 0.75))
    t_high = float(np.clip(t_base + 0.01, 0.55, 0.93))

    for i in range(N):
        pi = probs[i]
        area_mid = float((pi > t_mid).sum()) / total_px
        area_low = float((pi > t_low).sum())
        area_high = float((pi > t_high).sum())
        rho = area_low / max(area_high, 1.0)

        if rho > 1.65 or area_mid > 0.45:
            t_vec[i] = max(0.45, t_base - dt_high)
        elif rho > 1.32 or area_mid > 0.28:
            t_vec[i] = max(0.50, t_base - dt_mid)
        else:
            t_vec[i] = min(0.94, t_base + dt_crisp)
    return t_vec


def calibrate_bidirectional_alt_on_val(val_probs, val_gts):
    """
    Calibrates T_base in [0.35, 0.93] (removing the 0.85 ceiling!) and optimizes
    Scale-Anchored Bidirectional ALT offsets (dt_high, dt_mid, dt_crisp) on Official Val.
    """
    best_t_base, best_base_dice = 0.74, -1.0
    for t in np.linspace(0.35, 0.93, 59):
        d = compute_fast_val_dice(val_probs, val_gts, float(t), use_post=False)
        if d > best_base_dice:
            best_base_dice = d
            best_t_base = float(t)

    best_alt_dice = best_base_dice
    best_params = {
        "t_base": best_t_base,
        "dt_high": 0.0,
        "dt_mid": 0.0,
        "dt_crisp": 0.0,
        "use_post": False,
        "val_base_dice": best_base_dice,
        "val_alt_dice": best_base_dice,
    }

    for dt_h in [0.00, 0.06, 0.10, 0.14, 0.18, 0.22, 0.25]:
        for dt_m in [0.00, 0.03, 0.06, 0.09, 0.12]:
            if dt_m > dt_h:
                continue
            for dt_c in [0.00, 0.01, 0.02]:
                cand_p = {"t_base": best_t_base, "dt_high": dt_h, "dt_mid": dt_m, "dt_crisp": dt_c}
                t_vec = get_bi_alt_thresholds(val_probs, cand_p)
                d = compute_fast_val_dice(val_probs, val_gts, t_vec, use_post=False)
                if d > best_alt_dice:
                    best_alt_dice = d
                    best_params.update({
                        "dt_high": float(dt_h),
                        "dt_mid": float(dt_m),
                        "dt_crisp": float(dt_c),
                        "val_alt_dice": float(d),
                    })

    # If Val has few diffuse lesions so dt_high=0 ties, apply the validated prior if neutral on Val
    if best_params["dt_high"] == 0.0:
        cand_prior = {"t_base": best_t_base, "dt_high": 0.18, "dt_mid": 0.06, "dt_crisp": 0.0}
        t_vec_p = get_bi_alt_thresholds(val_probs, cand_prior)
        d_prior = compute_fast_val_dice(val_probs, val_gts, t_vec_p, use_post=False)
        if d_prior >= best_base_dice - 0.0004:
            best_params.update({"dt_high": 0.18, "dt_mid": 0.06, "dt_crisp": 0.0, "val_alt_dice": float(d_prior)})

    # Check if morphological post-processing improves Val Dice
    t_vec_best = get_bi_alt_thresholds(val_probs, best_params)
    d_post = compute_fast_val_dice(val_probs, val_gts, t_vec_best, use_post=True)
    if d_post > best_params["val_alt_dice"]:
        best_params["use_post"] = True
        best_params["val_alt_dice"] = float(d_post)

    return best_params


def build_decoupled_soup_state_dict(sd_list, enc_weights, dec_weights):
    out_sd = {}
    for k in sd_list[0].keys():
        if sd_list[0][k].dtype.is_floating_point:
            w_vec = enc_weights if k.startswith("backbone.") else dec_weights
            acc = torch.zeros_like(sd_list[0][k], dtype=torch.float32)
            for idx, sd in enumerate(sd_list):
                if w_vec[idx] != 0.0:
                    acc.add_(sd[k].float(), alpha=float(w_vec[idx]))
            out_sd[k] = acc.to(sd_list[0][k].dtype)
        else:
            out_sd[k] = sd_list[0][k].clone()
    return out_sd


@torch.no_grad()
def recalibrate_bn_stats(model, train_clean_loader, device, num_batches=50):
    model.train()
    for m in model.modules():
        if hasattr(m, "drop_prob"):
            m.drop_prob = 0.0
    for b_idx, (imgs, _, _) in enumerate(train_clean_loader):
        if b_idx >= num_batches:
            break
        imgs = imgs.to(device, non_blocking=True)
        with autocast("cuda"):
            _ = model(imgs)
    model.eval()


def evaluate_predictions(probs, gts, names, thresh_vec, mode_name, out_dir, use_post=False):
    records = []
    global_inter, global_union = 0.0, 0.0
    if np.ndim(thresh_vec) == 0:
        thresh_vec = np.full(len(probs), float(thresh_vec), dtype=np.float32)

    for idx in range(len(probs)):
        t_i = float(thresh_vec[idx])
        pred_mask = (probs[idx] > t_i).astype(np.uint8)
        if use_post:
            pred_mask = post_process_prediction(pred_mask)
        gt_mask = gts[idx].astype(np.uint8)

        p_f = pred_mask.flatten()
        g_f = gt_mask.flatten()
        tp = float(np.sum((p_f == 1) & (g_f == 1)))
        fp = float(np.sum((p_f == 1) & (g_f == 0)))
        fn = float(np.sum((p_f == 0) & (g_f == 1)))
        tn = float(np.sum((p_f == 0) & (g_f == 0)))

        global_inter += tp
        global_union += (tp + fp + tp + fn)

        dice = (2.0 * tp + 1e-7) / (2.0 * tp + fp + fn + 1e-7)
        iou = (tp + 1e-7) / (tp + fp + fn + 1e-7)
        j_thresh = iou if iou >= 0.65 else 0.0
        sens = (tp + 1e-7) / (tp + fn + 1e-7)
        spec = (tn + 1e-7) / (tn + fp + 1e-7)
        prec = (tp + 1e-7) / (tp + fp + 1e-7)
        fpr = (fp + 1e-7) / (fp + tn + 1e-7)
        fnr = (fn + 1e-7) / (fn + tp + 1e-7)

        hd95, asd, bf2 = compute_surface_distances(pred_mask, gt_mask)
        records.append({
            "image": names[idx], "threshold": t_i,
            "Dice": float(dice), "mIoU": float(iou), "J_thresh": float(j_thresh),
            "Sensitivity": float(sens), "Specificity": float(spec),
            "Precision": float(prec), "FPR": float(fpr), "FNR": float(fnr),
            "HD95": float(hd95), "ASD": float(asd), "BF2": float(bf2)
        })

    df_raw = pd.DataFrame(records)
    df_raw.to_csv(os.path.join(out_dir, f"benchmark_raw_{mode_name}.csv"), index=False)

    global_micro_dice = (2.0 * global_inter) / (global_union + 1e-7)
    failed_cases = int((df_raw["mIoU"] < 0.65).sum())

    metrics = ["Dice", "mIoU", "J_thresh", "Sensitivity", "Specificity", "Precision", "FPR", "FNR", "HD95", "ASD", "BF2"]
    summary_rows = []
    for m in metrics:
        arr = df_raw[m].values
        summary_rows.append({
            "Metric": m,
            "Mean": float(np.mean(arr)),
            "Std": float(np.std(arr)),
            "Median": float(np.median(arr)),
            "IQR": float(np.percentile(arr, 75) - np.percentile(arr, 25)),
            "Min": float(np.min(arr)),
            "Max": float(np.max(arr))
        })
    df_sum = pd.DataFrame(summary_rows)
    df_sum.to_csv(os.path.join(out_dir, f"benchmark_summary_{mode_name}.csv"), index=False)

    mean_dice = float(df_raw["Dice"].mean())
    median_dice = float(df_raw["Dice"].median())
    mean_iou = float(df_raw["mIoU"].mean())
    mean_j_thresh = float(df_raw["J_thresh"].mean())
    mean_sens = float(df_raw["Sensitivity"].mean())
    mean_prec = float(df_raw["Precision"].mean())
    mean_spec = float(df_raw["Specificity"].mean())
    mean_hd95 = float(df_raw["HD95"].mean())
    mean_asd = float(df_raw["ASD"].mean())

    print(f"\n================ {mode_name} (N={len(df_raw)}) ================", flush=True)
    print(f"  Mean Threshold    : {float(np.mean(thresh_vec)):.4f}", flush=True)
    print(f"  Macro Mean Dice   : {mean_dice:.4f} (Median: {median_dice:.4f})", flush=True)
    print(f"  Global Micro Dice : {global_micro_dice:.4f}", flush=True)
    print(f"  mIoU (Jaccard)    : {mean_iou:.4f} (ISIC J_thresh: {mean_j_thresh:.4f})", flush=True)
    print(f"  Sensitivity       : {mean_sens:.4f} | Precision: {mean_prec:.4f} | Specificity: {mean_spec:.4f}", flush=True)
    print(f"  HD95              : {mean_hd95:.2f} px | ASD: {mean_asd:.2f} px", flush=True)
    print(f"  Failed (J<0.65)   : {failed_cases} / {len(df_raw)}", flush=True)
    return {
        "Mode": mode_name,
        "Mean Threshold": float(np.mean(thresh_vec)),
        "Macro Dice": mean_dice,
        "Median Dice": median_dice,
        "Global Micro Dice": float(global_micro_dice),
        "mIoU": mean_iou,
        "J_thresh": mean_j_thresh,
        "Sensitivity": mean_sens,
        "Precision": mean_prec,
        "Specificity": mean_spec,
        "HD95": mean_hd95,
        "ASD": mean_asd,
        "Failed Cases (J<0.65)": failed_cases
    }


def build_llrd_param_groups(model, base_lr=4.5e-5, weight_decay=1e-4, llrd_decay=0.65):
    group_map = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if not name.startswith("backbone."):
            g_name = "head_decoder"
            lr = base_lr
        elif "stages.3" in name or "Stages.3" in name or ".3." in name:
            g_name = "stage3"
            lr = base_lr * (llrd_decay ** 1)
        elif "stages.2" in name or "Stages.2" in name or ".2." in name:
            g_name = "stage2"
            lr = base_lr * (llrd_decay ** 2)
        elif "stages.1" in name or "Stages.1" in name or ".1." in name:
            g_name = "stage1"
            lr = base_lr * (llrd_decay ** 3)
        else:
            g_name = "stem_stage0"
            lr = base_lr * (llrd_decay ** 4)

        no_decay = (param.ndim == 1) or name.endswith(".bias") or ("norm" in name.lower()) or ("bn" in name.lower())
        wd = 0.0 if no_decay else weight_decay
        key = (g_name, lr, wd)
        if key not in group_map:
            group_map[key] = []
        group_map[key].append(param)

    param_groups = []
    for (g_name, lr, wd), params in group_map.items():
        param_groups.append({"params": params, "lr": lr, "weight_decay": wd, "group_name": g_name})
    return param_groups


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Running GDF-Net: Scale-Anchored Bi-ALT, Weight-Space Model Soup, 8-View GeM Ensemble & Real-Time Knowledge Distillation on Device: {device}", flush=True)
    if torch.cuda.is_available():
        print(f"[*] GPU: {torch.cuda.get_device_name(0)}", flush=True)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True

    out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "./results_pa2_plus"
    os.makedirs(out_dir, exist_ok=True)
    DATA_ROOT = "/kaggle/temp/isic-2018" if os.path.exists("/kaggle") else "./isic-2018"
    os.makedirs(DATA_ROOT, exist_ok=True)

    def find_dir_by_name(target_name, search_roots):
        for base in search_roots:
            if not os.path.exists(base):
                continue
            for root, dirs, files in os.walk(base):
                if os.path.basename(root).lower() == target_name.lower():
                    return root
        return None

    def _unzip_all(root):
        for z in glob.glob(os.path.join(root, "*.zip")):
            print(f"[*] Extracting: {os.path.basename(z)}...", flush=True)
            try:
                with zipfile.ZipFile(z) as f:
                    f.extractall(root)
                os.remove(z)
            except Exception as e:
                print(f"[!] Warning while unzipping {z}: {e}", flush=True)

    search_roots = ["/kaggle/input", DATA_ROOT]
    train_img_dir = find_dir_by_name("ISIC2018_Task1-2_Training_Input", search_roots)
    train_gt_dir  = find_dir_by_name("ISIC2018_Task1_Training_GroundTruth", search_roots)
    val_img_dir   = find_dir_by_name("ISIC2018_Task1-2_Validation_Input", search_roots)
    val_gt_dir    = find_dir_by_name("ISIC2018_Task1_Validation_GroundTruth", search_roots)
    test_img_dir  = find_dir_by_name("ISIC2018_Task1-2_Test_Input", search_roots)
    test_gt_dir   = find_dir_by_name("ISIC2018_Task1_Test_GroundTruth", search_roots)

    if not test_gt_dir:
        subprocess.run(["kaggle", "datasets", "download", "-d", "mfahimhossen/isic2018-test-groundtruth", "-p", DATA_ROOT], check=True)
        _unzip_all(DATA_ROOT)
    if not (train_img_dir and test_img_dir):
        subprocess.run(["kaggle", "datasets", "download", "-d", "tschandl/isic2018-challenge-task1-data-segmentation", "-p", DATA_ROOT], check=True)
        _unzip_all(DATA_ROOT)
    if not val_img_dir:
        subprocess.run(["wget", "-q", "-O", f"{DATA_ROOT}/val_input.zip", "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Validation_Input.zip"], check=True)
        _unzip_all(DATA_ROOT)
    if not val_gt_dir:
        subprocess.run(["wget", "-q", "-O", f"{DATA_ROOT}/val_gt.zip", "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1_Validation_GroundTruth.zip"], check=True)
        _unzip_all(DATA_ROOT)

    train_img_dir = find_dir_by_name("ISIC2018_Task1-2_Training_Input", search_roots)
    train_gt_dir  = find_dir_by_name("ISIC2018_Task1_Training_GroundTruth", search_roots)
    val_img_dir   = find_dir_by_name("ISIC2018_Task1-2_Validation_Input", search_roots)
    val_gt_dir    = find_dir_by_name("ISIC2018_Task1_Validation_GroundTruth", search_roots)
    test_img_dir  = find_dir_by_name("ISIC2018_Task1-2_Test_Input", search_roots)
    test_gt_dir   = find_dir_by_name("ISIC2018_Task1_Test_GroundTruth", search_roots)

    def isic_id(path):
        base = os.path.basename(path).replace("_segmentation", "")
        return os.path.splitext(base)[0]

    def pair_data(img_dir, gt_dir):
        imgs = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        msks = sorted(glob.glob(os.path.join(gt_dir, "*.png")))
        id_map = {isic_id(m): m for m in msks}
        paired_x, paired_y = [], []
        for im in imgs:
            imid = isic_id(im)
            if imid in id_map:
                paired_x.append(im)
                paired_y.append(id_map[imid])
        return paired_x, paired_y

    train_x_raw, train_y_raw = pair_data(train_img_dir, train_gt_dir)
    val_x_raw, val_y_raw     = pair_data(val_img_dir, val_gt_dir)
    test_x_raw, test_y_raw   = pair_data(test_img_dir, test_gt_dir)
    print(f"[*] Split Sizes: Train={len(train_x_raw)} | Val={len(val_x_raw)} | Blind Test={len(test_x_raw)}", flush=True)

    CACHE_ROOT = os.path.join(DATA_ROOT, "fast_ssd_cache")
    train_imgs, train_msks = prepare_fast_ssd_cache(train_x_raw, train_y_raw, os.path.join(CACHE_ROOT, "train_384"), IMG_SIZE, desc="Train Cache")
    val_imgs, val_msks     = prepare_fast_ssd_cache(val_x_raw, val_y_raw, os.path.join(CACHE_ROOT, "val_384"), IMG_SIZE, desc="Val Cache")
    test_imgs, test_msks   = prepare_fast_ssd_cache(test_x_raw, test_y_raw, os.path.join(CACHE_ROOT, "test_384"), IMG_SIZE, desc="Test Cache")

    train_loader = DataLoader(FastISICDataset(train_imgs, train_msks, is_train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    train_clean_loader = DataLoader(FastISICDataset(train_imgs, train_msks, is_train=False), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(FastISICDataset(val_imgs, val_msks, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(FastISICDataset(test_imgs, test_msks, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    master_rows = []

    ckpt_map = find_checkpoints()
    m1_path = ckpt_map["M1_Canonical_300ep"]
    m2_path = ckpt_map["M2_Canonical_60ep_MS"]
    m3_path = ckpt_map["M3_Feedback_25ep"]
    m4_path = ckpt_map["M4_Canonical_AsymTversky"]

    if m1_path is None:
        raise FileNotFoundError("Could not find M1_Canonical_300ep checkpoint!")

    def load_clean_sd(path):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        if isinstance(ckpt, dict):
            if "model_state" in ckpt:
                sd = ckpt["model_state"]
            elif "model" in ckpt:
                sd = ckpt["model"]
            else:
                sd = ckpt
        else:
            sd = ckpt
        return {k.replace("module.", ""): v for k, v in sd.items()}

    sd_m1 = load_clean_sd(m1_path)
    sd_m2 = load_clean_sd(m2_path) if m2_path else copy.deepcopy(sd_m1)
    sd_m4 = load_clean_sd(m4_path) if m4_path else copy.deepcopy(sd_m1)

    single_model = CanonicalGDFNet(backbone_name=BACKBONE_NAME, pretrained=False, drop_path_rate=0.15).to(device)
    total_params = sum(p.numel() for p in single_model.parameters())
    print(f"[*] Single Pure-CNN Student/Eval Model Params: {total_params / 1e6:.2f}M", flush=True)

    # ====================================================================
    # STAGE 1 (GDF-Net Weight-Soup): ZERO-RETRAIN SINGLE-MODEL OPTIMIZATION
    # ====================================================================
    print("\n=========================================================================", flush=True)
    print("STAGE 1: Scale-Anchored Bi-ALT, Weight-Space Model Soup (0.70 M1 + 0.30 M4) & 8-View GeM Ensemble", flush=True)
    print("=========================================================================", flush=True)

    single_model.load_state_dict(sd_m1, strict=True)
    v384_val_m1, v448_val_m1, v352_val_m1, val_gts, _ = extract_single_model_views(single_model, val_loader, device)
    v384_te_m1, v448_te_m1, v352_te_m1, test_gts, test_names = extract_single_model_views(single_model, test_loader, device)

    scale_weight_grid = [
        (4.0/6.0, 2.0/6.0, 0.0),
        (0.60, 0.30, 0.10),
        (0.55, 0.35, 0.10),
        (0.50, 0.35, 0.15),
    ]

    best_m1_gem_p, best_m1_sw, best_m1_val_d = 1.0, (4.0/6.0, 2.0/6.0, 0.0), -1.0
    for gp in [1.0, 1.25, 1.5, 1.75]:
        for sw in scale_weight_grid:
            vp = fuse_single_model_views(v384_val_m1, v448_val_m1, v352_val_m1, mode="ms8_gem", gem_p=gp, scale_weights=sw)
            alt_p = calibrate_bidirectional_alt_on_val(vp, val_gts)
            if alt_p["val_alt_dice"] > best_m1_val_d:
                best_m1_val_d = alt_p["val_alt_dice"]
                best_m1_gem_p = gp
                best_m1_sw = sw
                best_m1_alt = alt_p

    print(f"[*] GDF-Net Weight-Soup Baseline M1 (300ep) Optimal GeM_p={best_m1_gem_p}, ScaleWeights={tuple(round(x,2) for x in best_m1_sw)}", flush=True)
    print(f"    Bi-ALT: T_base={best_m1_alt['t_base']:.2f}, dT=(-high:{best_m1_alt['dt_high']:.2f}, -mid:{best_m1_alt['dt_mid']:.2f}, +crisp:{best_m1_alt['dt_crisp']:.2f}), Post={best_m1_alt['use_post']} -> Val Dice={best_m1_val_d:.4f}", flush=True)

    # Mode 1: Baseline M1 Single-View + Bi-ALT
    val_sv_m1 = fuse_single_model_views(v384_val_m1, v448_val_m1, v352_val_m1, mode="sv")
    alt_sv_m1 = calibrate_bidirectional_alt_on_val(val_sv_m1, val_gts)
    te_sv_m1 = fuse_single_model_views(v384_te_m1, v448_te_m1, v352_te_m1, mode="sv")
    t_vec_m1_sv = get_bi_alt_thresholds(te_sv_m1, alt_sv_m1)
    r1 = evaluate_predictions(te_sv_m1, test_gts, test_names, t_vec_m1_sv, "GDF-Net (+Bi-ALT, T_base=0.74, Single-View)", out_dir, use_post=alt_sv_m1["use_post"])
    r1["Description"] = f"1. GDF-Net Weight-Soup Baseline Single S18 (300ep) Single-View + Bi-ALT (T_base={alt_sv_m1['t_base']:.2f}, No TTA)"
    master_rows.append(r1)

    # Mode 2: Baseline M1 8-View GeM MS-TTA + Bi-ALT
    te_ms8_m1 = fuse_single_model_views(v384_te_m1, v448_te_m1, v352_te_m1, mode="ms8_gem", gem_p=best_m1_gem_p, scale_weights=best_m1_sw)
    t_vec_m1_ms8 = get_bi_alt_thresholds(te_ms8_m1, best_m1_alt)
    r2 = evaluate_predictions(te_ms8_m1, test_gts, test_names, t_vec_m1_ms8, "GDF-Net (Base + 8-View GeM MS-TTA + Bi-ALT)", out_dir, use_post=best_m1_alt["use_post"])
    r2["Description"] = f"2. GDF-Net Weight-Soup Baseline Single S18 (300ep) 8-View GeM(p={best_m1_gem_p}) MS-TTA + Bi-ALT"
    master_rows.append(r2)

    # 2. Search 3-Way Stage-Decoupled Weight Soup on Official Validation
    print("\n[*] Searching 3-Way Stage-Decoupled Weight Soup (M1_300ep + M2_60ep_MS + M4_AsymTversky) on 100 Val...", flush=True)
    soup_candidates = [
        ("Uniform_M1", (1.00, 0.00, 0.00), (1.00, 0.00, 0.00)),
        ("WiSE_80_00_20", (0.80, 0.00, 0.20), (0.80, 0.00, 0.20)),
        ("WiSE_70_00_30", (0.70, 0.00, 0.30), (0.70, 0.00, 0.30)),
        ("Soup_80_10_10", (0.80, 0.10, 0.10), (0.80, 0.10, 0.10)),
        ("Soup_75_15_10", (0.75, 0.15, 0.10), (0.75, 0.15, 0.10)),
        ("Soup_70_15_15", (0.70, 0.15, 0.15), (0.70, 0.15, 0.15)),
        ("Soup_65_15_20", (0.65, 0.15, 0.20), (0.65, 0.15, 0.20)),
        ("Decoupled_EncRecall_DecSharp_1", (0.60, 0.15, 0.25), (0.85, 0.05, 0.10)),
        ("Decoupled_EncRecall_DecSharp_2", (0.65, 0.10, 0.25), (0.85, 0.10, 0.05)),
        ("Decoupled_EncSharp_DecRecall_1", (0.85, 0.05, 0.10), (0.65, 0.15, 0.20)),
    ]

    best_soup_val_dice = -1.0
    best_soup_sd = copy.deepcopy(sd_m1)
    best_soup_name = "Uniform_M1"

    for c_name, enc_w, dec_w in soup_candidates:
        cand_sd = build_decoupled_soup_state_dict([sd_m1, sd_m2, sd_m4], enc_w, dec_w)
        single_model.load_state_dict(cand_sd, strict=True)
        v384_v, v448_v, v352_v, _, _ = extract_single_model_views(single_model, val_loader, device)
        vp = fuse_single_model_views(v384_v, v448_v, v352_v, mode="ms8_gem", gem_p=1.25, scale_weights=(4.0/6.0, 2.0/6.0, 0.0))
        alt_c = calibrate_bidirectional_alt_on_val(vp, val_gts)
        print(f"    [Soup Candidate {c_name:30s}] Enc={enc_w} Dec={dec_w} -> Val Bi-ALT Dice={alt_c['val_alt_dice']:.4f} (T_base={alt_c['t_base']:.2f})", flush=True)
        if alt_c["val_alt_dice"] > best_soup_val_dice:
            best_soup_val_dice = alt_c["val_alt_dice"]
            best_soup_sd = copy.deepcopy(cand_sd)
            best_soup_name = c_name

    # Check if AdaBN improves the winning 3-Way Soup on Validation
    single_model.load_state_dict(best_soup_sd, strict=True)
    recalibrate_bn_stats(single_model, train_clean_loader, device, num_batches=50)
    v384_v_bn, v448_v_bn, v352_v_bn, _, _ = extract_single_model_views(single_model, val_loader, device)
    vp_bn = fuse_single_model_views(v384_v_bn, v448_v_bn, v352_v_bn, mode="ms8_gem", gem_p=1.25, scale_weights=(4.0/6.0, 2.0/6.0, 0.0))
    alt_bn = calibrate_bidirectional_alt_on_val(vp_bn, val_gts)
    print(f"    [AdaBN Check on {best_soup_name}] Val Bi-ALT Dice with AdaBN={alt_bn['val_alt_dice']:.4f} vs Original BN={best_soup_val_dice:.4f}", flush=True)
    if alt_bn["val_alt_dice"] > best_soup_val_dice:
        best_soup_val_dice = alt_bn["val_alt_dice"]
        best_soup_sd = copy.deepcopy(single_model.state_dict())
        best_soup_name += "+AdaBN"

    print(f"[*] Best GDF-Net Weight-Soup Single-Model Weight Soup: {best_soup_name} (Val Dice = {best_soup_val_dice:.4f})", flush=True)
    single_model.load_state_dict(best_soup_sd, strict=True)
    v384_val_soup, v448_val_soup, v352_val_soup, _, _ = extract_single_model_views(single_model, val_loader, device)
    v384_te_soup, v448_te_soup, v352_te_soup, _, _ = extract_single_model_views(single_model, test_loader, device)

    # Mode 3: 3-Way Decoupled Soup Single-View (No TTA, 31.3 FPS) + Bi-ALT
    val_sv_soup = fuse_single_model_views(v384_val_soup, v448_val_soup, v352_val_soup, mode="sv")
    alt_sv_soup = calibrate_bidirectional_alt_on_val(val_sv_soup, val_gts)
    te_sv_soup = fuse_single_model_views(v384_te_soup, v448_te_soup, v352_te_soup, mode="sv")
    r3 = evaluate_predictions(te_sv_soup, test_gts, test_names, get_bi_alt_thresholds(te_sv_soup, alt_sv_soup), "GDF-Net (Soup, SV No KD, T_base=0.89)", out_dir, use_post=alt_sv_soup["use_post"])
    r3["Description"] = f"3. GDF-Net Weight-Soup 3-Way Weight Soup ({best_soup_name}) Single-View + Bi-ALT (T_base={alt_sv_soup['t_base']:.2f}, No TTA)"
    master_rows.append(r3)

    # Mode 4: 3-Way Decoupled Soup 4-View TTA + Bi-ALT
    val_tta4_soup = fuse_single_model_views(v384_val_soup, v448_val_soup, v352_val_soup, mode="tta4", gem_p=1.25)
    alt_tta4_soup = calibrate_bidirectional_alt_on_val(val_tta4_soup, val_gts)
    te_tta4_soup = fuse_single_model_views(v384_te_soup, v448_te_soup, v352_te_soup, mode="tta4", gem_p=1.25)
    r4 = evaluate_predictions(te_tta4_soup, test_gts, test_names, get_bi_alt_thresholds(te_tta4_soup, alt_tta4_soup), "GDF-Net (Soup, 4-View TTA + Bi-ALT)", out_dir, use_post=alt_tta4_soup["use_post"])
    r4["Description"] = f"4. GDF-Net Weight-Soup 3-Way Weight Soup ({best_soup_name}) 4-View GeM TTA + Bi-ALT (T_base={alt_tta4_soup['t_base']:.2f})"
    master_rows.append(r4)

    # GDF-Net (Soup / Ensemble): 3-Way Decoupled Soup 8-View GeM MS-TTA + Bi-ALT
    best_soup_gp, best_soup_sw, best_soup_ms_d = 1.25, (4.0/6.0, 2.0/6.0, 0.0), -1.0
    for gp in [1.0, 1.25, 1.5, 1.75]:
        for sw in scale_weight_grid:
            vp = fuse_single_model_views(v384_val_soup, v448_val_soup, v352_val_soup, mode="ms8_gem", gem_p=gp, scale_weights=sw)
            alt_p = calibrate_bidirectional_alt_on_val(vp, val_gts)
            if alt_p["val_alt_dice"] > best_soup_ms_d:
                best_soup_ms_d = alt_p["val_alt_dice"]
                best_soup_gp = gp
                best_soup_sw = sw
                best_soup_ms_alt = alt_p

    te_ms8_soup = fuse_single_model_views(v384_te_soup, v448_te_soup, v352_te_soup, mode="ms8_gem", gem_p=best_soup_gp, scale_weights=best_soup_sw)
    r5 = evaluate_predictions(te_ms8_soup, test_gts, test_names, get_bi_alt_thresholds(te_ms8_soup, best_soup_ms_alt), "GDF-Net (Ensemble, 8V MS-TTA + Bi-ALT, T_base=0.89)", out_dir, use_post=best_soup_ms_alt["use_post"])
    r5["Description"] = f"5. GDF-Net Weight-Soup 3-Way Weight Soup ({best_soup_name}) 8-View GeM(p={best_soup_gp}) MS-TTA + Bi-ALT (T_base={best_soup_ms_alt['t_base']:.2f})"
    master_rows.append(r5)

    # ====================================================================
    # STAGE 2 (GDF-Net Real-Time KD): PA1-TEACHER ENSEMBLE-TO-SINGLE SELF-DISTILLATION
    # ====================================================================
    print("\n=========================================================================", flush=True)
    print("STAGE 2: Online 4-Teacher (M1-M4) Ensemble Knowledge Distillation -> GDF-Net (Real-Time, SV KD)", flush=True)
    print("=========================================================================", flush=True)

    teacher_m1 = CanonicalGDFNet(backbone_name=BACKBONE_NAME, pretrained=False, drop_path_rate=0.0).to(device)
    teacher_m1.load_state_dict(sd_m1, strict=True)
    teacher_m1.eval()
    for p in teacher_m1.parameters():
        p.requires_grad_(False)

    teacher_m2 = CanonicalGDFNet(backbone_name=BACKBONE_NAME, pretrained=False, drop_path_rate=0.0).to(device)
    teacher_m2.load_state_dict(sd_m2, strict=True)
    teacher_m2.eval()
    for p in teacher_m2.parameters():
        p.requires_grad_(False)

    teacher_m4 = CanonicalGDFNet(backbone_name=BACKBONE_NAME, pretrained=False, drop_path_rate=0.0).to(device)
    teacher_m4.load_state_dict(sd_m4, strict=True)
    teacher_m4.eval()
    for p in teacher_m4.parameters():
        p.requires_grad_(False)

    teacher_m3 = None
    if m3_path is not None:
        sd_m3 = load_clean_sd(m3_path)
        teacher_m3 = FeedbackGDFNet(backbone_name=BACKBONE_NAME, pretrained=False, drop_path_rate=0.0).to(device)
        teacher_m3.load_state_dict(sd_m3, strict=True)
        teacher_m3.eval()
        for p in teacher_m3.parameters():
            p.requires_grad_(False)
        print("[*] Loaded M3 FeedbackGDFNet Teacher successfully (Full 4-Model 4-Teacher (M1-M4) Ensemble Ready!).", flush=True)

    single_model.load_state_dict(best_soup_sd, strict=True)
    ema = ModelEMA(single_model, decay=0.996)

    criterion = EnsembleDistillationLoss(alpha_fp=0.42, beta_fn=0.58, gamma=1.25, pos_weight=1.18, kd_weight=0.55, temp=2.0)
    param_groups = build_llrd_param_groups(single_model, base_lr=4.5e-5, weight_decay=1e-4, llrd_decay=0.65)
    optimizer = optim.AdamW(param_groups)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=DISTILL_EPOCHS, eta_min=1e-6)
    scaler = GradScaler("cuda")

    best_kd_val_dice = best_soup_val_dice
    best_kd_sd = copy.deepcopy(best_soup_sd)
    history_rows = []

    for epoch in range(1, DISTILL_EPOCHS + 1):
        t0 = time.time()
        single_model.train()
        running_loss = 0.0

        for imgs, msks, _ in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            msks = msks.to(device, non_blocking=True)

            with torch.no_grad(), autocast("cuda"):
                z1 = teacher_m1(imgs)
                z2 = teacher_m2(imgs)
                z4 = teacher_m4(imgs)
                p1_t = torch.clamp(torch.sigmoid(z1), 1e-6, 1.0 - 1e-6)
                p2_t = torch.clamp(torch.sigmoid(z2), 1e-6, 1.0 - 1e-6)
                p4_t = torch.clamp(torch.sigmoid(z4), 1e-6, 1.0 - 1e-6)

                if teacher_m3 is not None:
                    z3 = teacher_m3(imgs)
                    p3_t = torch.clamp(torch.sigmoid(z3), 1e-6, 1.0 - 1e-6)
                    gem_p = 1.5
                    teacher_prob = torch.pow(
                        0.38 * torch.pow(p1_t, gem_p) +
                        0.14 * torch.pow(p2_t, gem_p) +
                        0.36 * torch.pow(p3_t, gem_p) +
                        0.12 * torch.pow(p4_t, gem_p),
                        1.0 / gem_p
                    )
                    teacher_logits = 0.38 * z1 + 0.14 * z2 + 0.36 * z3 + 0.12 * z4
                else:
                    gem_p = 1.5
                    teacher_prob = torch.pow(
                        0.55 * torch.pow(p1_t, gem_p) +
                        0.25 * torch.pow(p2_t, gem_p) +
                        0.20 * torch.pow(p4_t, gem_p),
                        1.0 / gem_p
                    )
                    teacher_logits = 0.55 * z1 + 0.25 * z2 + 0.20 * z4

            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda"):
                outputs = single_model(imgs, return_deep_supervision=True)
                loss = criterion(outputs, msks, teacher_prob, teacher_logits)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(single_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            ema.update(single_model)
            running_loss += loss.item()

        scheduler.step()
        avg_loss = running_loss / len(train_loader)

        single_model.eval()
        v384_r, v448_r, v352_r, _, _ = extract_single_model_views(single_model, val_loader, device)
        vp_r = fuse_single_model_views(v384_r, v448_r, v352_r, mode="ms8_gem", gem_p=1.25, scale_weights=(4.0/6.0, 2.0/6.0, 0.0))
        alt_r = calibrate_bidirectional_alt_on_val(vp_r, val_gts)

        v384_e, v448_e, v352_e, _, _ = extract_single_model_views(ema.ema, val_loader, device)
        vp_e = fuse_single_model_views(v384_e, v448_e, v352_e, mode="ms8_gem", gem_p=1.25, scale_weights=(4.0/6.0, 2.0/6.0, 0.0))
        alt_e = calibrate_bidirectional_alt_on_val(vp_e, val_gts)

        dt = time.time() - t0
        cur_best = max(alt_r["val_alt_dice"], alt_e["val_alt_dice"])
        tag = ""
        if cur_best > best_kd_val_dice:
            best_kd_val_dice = cur_best
            best_kd_sd = copy.deepcopy(ema.ema.state_dict() if alt_e["val_alt_dice"] >= alt_r["val_alt_dice"] else single_model.state_dict())
            tag = " [NEW BEST KD SINGLE MODEL]"

        print(f"Epoch [{epoch:02d}/{DISTILL_EPOCHS:02d}] ({dt:.1f}s) | Loss: {avg_loss:.4f} | Val Raw MS8-BiALT: {alt_r['val_alt_dice']:.4f} (T={alt_r['t_base']:.2f}) | Val EMA MS8-BiALT: {alt_e['val_alt_dice']:.4f} (T={alt_e['t_base']:.2f}){tag}", flush=True)
        history_rows.append({
            "epoch": epoch, "loss": avg_loss,
            "val_raw_ms8_dice": alt_r["val_alt_dice"],
            "val_ema_ms8_dice": alt_e["val_alt_dice"],
            "best_kd_val_dice": best_kd_val_dice
        })

    pd.DataFrame(history_rows).to_csv(os.path.join(out_dir, f"training_history_{MODEL_TAG}.csv"), index=False)

    del teacher_m1, teacher_m2, teacher_m4, teacher_m3
    torch.cuda.empty_cache()

    # Evaluate Pure Distilled Student (best_kd_sd) on 1,000 Blind Test
    single_model.load_state_dict(best_kd_sd, strict=True)
    v384_val_kd, v448_val_kd, v352_val_kd, _, _ = extract_single_model_views(single_model, val_loader, device)
    v384_te_kd, v448_te_kd, v352_te_kd, _, _ = extract_single_model_views(single_model, test_loader, device)

    # GDF-Net (Real-Time KD): GDF-Net Real-Time KD Distilled Single S18 — Single-View (No TTA, 31.3 FPS) + Bi-ALT
    val_sv_kd = fuse_single_model_views(v384_val_kd, v448_val_kd, v352_val_kd, mode="sv")
    alt_sv_kd = calibrate_bidirectional_alt_on_val(val_sv_kd, val_gts)
    te_sv_kd = fuse_single_model_views(v384_te_kd, v448_te_kd, v352_te_kd, mode="sv")
    r6 = evaluate_predictions(te_sv_kd, test_gts, test_names, get_bi_alt_thresholds(te_sv_kd, alt_sv_kd), "GDF-Net (Real-Time, SV KD + Bi-ALT, T_base=0.91)", out_dir, use_post=alt_sv_kd["use_post"])
    r6["Description"] = f"6. GDF-Net Real-Time KD PA1-Distilled Single S18 (29.50M) Single-View + Bi-ALT (T_base={alt_sv_kd['t_base']:.2f}, No TTA)"
    master_rows.append(r6)

    # GDF-Net (Distilled 4V-TTA): GDF-Net Real-Time KD Distilled Single S18 — 4-View TTA + Bi-ALT
    val_tta4_kd = fuse_single_model_views(v384_val_kd, v448_val_kd, v352_val_kd, mode="tta4", gem_p=1.25)
    alt_tta4_kd = calibrate_bidirectional_alt_on_val(val_tta4_kd, val_gts)
    te_tta4_kd = fuse_single_model_views(v384_te_kd, v448_te_kd, v352_te_kd, mode="tta4", gem_p=1.25)
    r7 = evaluate_predictions(te_tta4_kd, test_gts, test_names, get_bi_alt_thresholds(te_tta4_kd, alt_tta4_kd), "GDF-Net (Distilled, 4-View TTA + Bi-ALT)", out_dir, use_post=alt_tta4_kd["use_post"])
    r7["Description"] = f"7. GDF-Net Real-Time KD PA1-Distilled Single S18 (29.50M) 4-View GeM TTA + Bi-ALT (T_base={alt_tta4_kd['t_base']:.2f})"
    master_rows.append(r7)

    # GDF-Net (Distilled 8V-TTA): GDF-Net Real-Time KD Distilled Single S18 — 8-View GeM MS-TTA + Bi-ALT
    best_kd_gp, best_kd_sw, best_kd_ms_d = 1.25, (4.0/6.0, 2.0/6.0, 0.0), -1.0
    for gp in [1.0, 1.25, 1.5, 1.75]:
        for sw in scale_weight_grid:
            vp = fuse_single_model_views(v384_val_kd, v448_val_kd, v352_val_kd, mode="ms8_gem", gem_p=gp, scale_weights=sw)
            alt_p = calibrate_bidirectional_alt_on_val(vp, val_gts)
            if alt_p["val_alt_dice"] > best_kd_ms_d:
                best_kd_ms_d = alt_p["val_alt_dice"]
                best_kd_gp = gp
                best_kd_sw = sw
                best_kd_ms_alt = alt_p

    te_ms8_kd = fuse_single_model_views(v384_te_kd, v448_te_kd, v352_te_kd, mode="ms8_gem", gem_p=best_kd_gp, scale_weights=best_kd_sw)
    r8 = evaluate_predictions(te_ms8_kd, test_gts, test_names, get_bi_alt_thresholds(te_ms8_kd, best_kd_ms_alt), "GDF-Net (Distilled, 8-View GeM MS-TTA + Bi-ALT)", out_dir, use_post=best_kd_ms_alt["use_post"])
    r8["Description"] = f"8. GDF-Net Real-Time KD PA1-Distilled Single S18 (29.50M) 8-View GeM(p={best_kd_gp}) MS-TTA + Bi-ALT (T_base={best_kd_ms_alt['t_base']:.2f})"
    master_rows.append(r8)

    # 3. Distilled-WiSE-FT: Weight-Space Interpolation between Best_Soup and Best_KD
    print("\n[*] Running Distilled-WiSE-FT Interpolation on 100 Validation Images...", flush=True)
    best_wise_kd_d = -1.0
    best_wise_kd_sd = copy.deepcopy(best_kd_sd)
    best_wise_lam = 1.0

    for lam in [0.0, 0.20, 0.35, 0.50, 0.65, 0.80, 1.0]:
        w_sd = build_decoupled_soup_state_dict([best_soup_sd, best_kd_sd], (1.0 - lam, lam), (1.0 - lam, lam))
        single_model.load_state_dict(w_sd, strict=True)
        v384_w, v448_w, v352_w, _, _ = extract_single_model_views(single_model, val_loader, device)
        vp_w = fuse_single_model_views(v384_w, v448_w, v352_w, mode="ms8_gem", gem_p=1.25, scale_weights=(4.0/6.0, 2.0/6.0, 0.0))
        alt_w = calibrate_bidirectional_alt_on_val(vp_w, val_gts)
        print(f"    [Distilled-WiSE-FT lambda={lam:.2f}] Val MS8 Bi-ALT Dice = {alt_w['val_alt_dice']:.4f} (T_base={alt_w['t_base']:.2f})", flush=True)
        if alt_w["val_alt_dice"] > best_wise_kd_d:
            best_wise_kd_d = alt_w["val_alt_dice"]
            best_wise_kd_sd = copy.deepcopy(w_sd)
            best_wise_lam = lam

    single_model.load_state_dict(best_wise_kd_sd, strict=True)
    v384_val_wkd, v448_val_wkd, v352_val_wkd, _, _ = extract_single_model_views(single_model, val_loader, device)
    v384_te_wkd, v448_te_wkd, v352_te_wkd, _, _ = extract_single_model_views(single_model, test_loader, device)

    # Mode 9: Distilled-WiSE-FT Single-View + Bi-ALT
    val_sv_wkd = fuse_single_model_views(v384_val_wkd, v448_val_wkd, v352_val_wkd, mode="sv")
    alt_sv_wkd = calibrate_bidirectional_alt_on_val(val_sv_wkd, val_gts)
    te_sv_wkd = fuse_single_model_views(v384_te_wkd, v448_te_wkd, v352_te_wkd, mode="sv")
    r9 = evaluate_predictions(te_sv_wkd, test_gts, test_names, get_bi_alt_thresholds(te_sv_wkd, alt_sv_wkd), "SingleModel_PA22_DistilledWiSE_SV_BiALT", out_dir, use_post=alt_sv_wkd["use_post"])
    r9["Description"] = f"9. GDF-Net Real-Time KD Distilled-WiSE-FT (lam={best_wise_lam:.2f}, 29.50M) Single-View + Bi-ALT (T_base={alt_sv_wkd['t_base']:.2f}, No TTA)"
    master_rows.append(r9)

    # Mode 10: Distilled-WiSE-FT 8-View GeM MS-TTA + Bi-ALT
    best_wkd_gp, best_wkd_sw, best_wkd_ms_d = 1.25, (4.0/6.0, 2.0/6.0, 0.0), -1.0
    for gp in [1.0, 1.25, 1.5, 1.75]:
        for sw in scale_weight_grid:
            vp = fuse_single_model_views(v384_val_wkd, v448_val_wkd, v352_val_wkd, mode="ms8_gem", gem_p=gp, scale_weights=sw)
            alt_p = calibrate_bidirectional_alt_on_val(vp, val_gts)
            if alt_p["val_alt_dice"] > best_wkd_ms_d:
                best_wkd_ms_d = alt_p["val_alt_dice"]
                best_wkd_gp = gp
                best_wkd_sw = sw
                best_wkd_ms_alt = alt_p

    te_ms8_wkd = fuse_single_model_views(v384_te_wkd, v448_te_wkd, v352_te_wkd, mode="ms8_gem", gem_p=best_wkd_gp, scale_weights=best_wkd_sw)
    r10 = evaluate_predictions(te_ms8_wkd, test_gts, test_names, get_bi_alt_thresholds(te_ms8_wkd, best_wkd_ms_alt), "SingleModel_PA22_DistilledWiSE_GeMMSTTA8_BiALT", out_dir, use_post=best_wkd_ms_alt["use_post"])
    r10["Description"] = f"10. GDF-Net Real-Time KD Distilled-WiSE-FT (lam={best_wise_lam:.2f}, 29.50M) 8-View GeM(p={best_wkd_gp}) MS-TTA + Bi-ALT (T_base={best_wkd_ms_alt['t_base']:.2f})"
    master_rows.append(r10)

    df_master = pd.DataFrame(master_rows)
    cols = ["Mode", "Description", "Mean Threshold", "Macro Dice", "Median Dice", "Global Micro Dice", "mIoU", "J_thresh", "Sensitivity", "Precision", "Specificity", "HD95", "ASD", "Failed Cases (J<0.65)"]
    df_master = df_master[cols]
    df_master.to_csv(os.path.join(out_dir, "master_single_pure_cnn_pa2_plus_distill_overview.csv"), index=False)

    print("\n=======================================================================================", flush=True)
    print("FINAL SINGLE-MODEL PURE-CNN (GDF-Net Weight-Soup + GDF-Net Real-Time KD) BENCHMARK SUMMARY ON 1,000 BLIND TEST", flush=True)
    print("=======================================================================================", flush=True)
    print(df_master.to_string(index=False), flush=True)


if __name__ == "__main__":
    main()


[*] Running GDF-Net: Scale-Anchored Bi-ALT, Weight-Space Model Soup, 8-View GeM Ensemble & Real-Time Knowledge Distillation on Device: cuda
[*] GPU: Tesla T4
[*] Split Sizes: Train=2594 | Val=100 | Blind Test=1000
[*] Caching Train Cache (2594 pairs -> 384x384)...
[+] Cache ready in 1287.6s!
[*] Caching Val Cache (100 pairs -> 384x384)...
[+] Cache ready in 28.6s!
[*] Caching Test Cache (1000 pairs -> 384x384)...
[+] Cache ready in 417.0s!
/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/kaggle/src/script.py:626: UserWarning: Argument(s) 'value, mask_value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(
[*] Found 16 .pth files across workspace:
    -> /kaggle/input/datasets/namnguynnnn/gdfnetv4-isic-pretrained/GDFNet_ConvFormerS18_ISIC2018_best.pth
    -> /kaggle/input/datasets/namnguynnnn/